## RAG Day 3

### Expert Question Answerer for InsureLLM

LangChain 1.0 implementation of a RAG pipeline.

Using the VectorStore we created last time (with HuggingFace `all-MiniLM-L6-v2`)

In [2]:
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain_ollama import ChatOllama
from langchain_chroma import Chroma
from langchain_core.messages import SystemMessage, HumanMessage
from langchain_huggingface import HuggingFaceEmbeddings
import gradio as gr

In [3]:
MODEL = "qwen3.5:9b" # "gpt-4.1-nano"
DB_NAME = "vector_db"
load_dotenv(override=True)

False

### Connect to Chroma; use Hugging Face all-MiniLM-L6-v2

In [4]:
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
vectorstore = Chroma(persist_directory=DB_NAME, embedding_function=embeddings)

### Set up the 2 key LangChain objects: retriever and llm

#### A sidebar on "temperature":
- Controls how diverse the output is
- A temperature of 0 means that the output should be predictable
- Higher temperature for more variety in answers

Some people describe temperature as being like 'creativity' but that's not quite right
- It actually controls which tokens get selected during inference
- temperature=0 means: always select the token with highest probability
- temperature=1 usually means: a token with 10% probability should be picked 10% of the time

Note: a temperature of 0 doesn't mean outputs will always be reproducible. You also need to set a random seed. We will do that in weeks 6-8. (Even then, it's not always reproducible.)

Note 2: if you want creativity, use the System Prompt!

In [13]:
# https://reference.langchain.com/python/langchain-core/vectorstores/base/VectorStore/as_retriever
retriever = vectorstore.as_retriever(search_kwargs={"k": 5})
llm = ChatOllama(temperature=0, model=MODEL)

### These LangChain objects implement the method `invoke()`

In [14]:
retriever.invoke("Who is Avery?")

[Document(id='f3f30244-9da8-4603-8f57-9496e10f0fe7', metadata={'source': 'knowledge-base/employees/Avery Lancaster.md', 'doc_type': 'employees'}, page_content="## Other HR Notes\n- **Professional Development**: Avery has actively participated in leadership training programs and industry conferences, representing Insurellm and fostering partnerships.  \n- **Diversity & Inclusion Initiatives**: Avery has championed a commitment to diversity in hiring practices, seeing visible improvements in team representation since 2021.  \n- **Work-Life Balance**: Feedback revealed concerns regarding work-life balance, which Avery has approached by implementing flexible working conditions and ensuring regular check-ins with the team.\n- **Community Engagement**: Avery led community outreach efforts, focusing on financial literacy programs, particularly aimed at underserved populations, improving Insurellm's corporate social responsibility image.  \n\nAvery Lancaster has demonstrated resilience and ada

In [8]:
llm.invoke("Who is Avery?")

AIMessage(content='The name "Avery" is quite common and could refer to many different people, places, or things! To give you the most helpful answer, could you clarify **which Avery** you\'re asking about? For example:  \n\n- A **celebrity** (actor, musician, etc.)?  \n- A **fictional character** from a book, movie, or TV show?  \n- A **historical figure**?  \n- Something else (e.g., a company, place, or concept)?  \n\nLet me know, and I\'ll do my best to help! 😊', additional_kwargs={}, response_metadata={'model': 'qwen3.5:9b', 'created_at': '2026-04-01T05:49:32.539736Z', 'done': True, 'done_reason': 'stop', 'total_duration': 93512202667, 'load_duration': 4511442917, 'prompt_eval_count': 14, 'prompt_eval_duration': 1772031709, 'eval_count': 1030, 'eval_duration': 84846886123, 'model_name': 'qwen3.5:9b', 'model_provider': 'ollama'}, id='lc_run--7ecb8fe5-7fff-49a6-a9a6-d51bcb1b609a-0', usage_metadata={'input_tokens': 14, 'output_tokens': 1030, 'total_tokens': 1044})

## Time to put this together!

In [15]:
SYSTEM_PROMPT_TEMPLATE = """
You are a knowledgeable, friendly assistant representing the company Insurellm.
You are chatting with a user about Insurellm.
If relevant, use the given context to answer any question.
If you don't know the answer, say so.
Context:
{context}
"""

In [16]:
def answer_question(question: str, history):
    docs = retriever.invoke(question)
    context = "\n\n".join(doc.page_content for doc in docs)
    system_prompt = SYSTEM_PROMPT_TEMPLATE.format(context=context)
    response = llm.invoke([SystemMessage(content=system_prompt), HumanMessage(content=question)])
    return response.content

In [17]:
answer_question("Who is Averi Lancaster?", [])

'Based on our company records, **Avery Lancaster** (often spelled as "Averi" in casual conversation) is the **Co-Founder and Chief Executive Officer (CEO)** of Insurellm.\n\nHere are a few key details about her role and background:\n\n*   **Leadership:** She co-founded Insurellm in 2015 and has guided the company to its current position as a leading Insurance Tech provider.\n*   **Location:** She is based in San Francisco, California.\n*   **Previous Experience:** Before launching Insurellm, she worked as a Senior Product Manager at Innovate Insurance Solutions (from 2013 to 2015), where she developed groundbreaking insurance products for the tech sector.\n*   **Expertise:** She is known for her innovative leadership strategies and risk management expertise that have helped catapult the company into the mainstream insurance market.\n\nShe is a valued asset to the company, with a current salary listed at $225,000.'

## What could possibly come next? 😂

In [ ]:
gr.ChatInterface(answer_question).launch()

## Admit it - you thought RAG would be more complicated than that!!